# Odlar iş masası: təlim

Bütün Odlar modelləri bu bir notebook ilə öyrədilir. Fərq yalnız config faylındadır (`configs/` qovluğu).

1. **Runtime → Change runtime type → T4 GPU** → Save (Kaggle-da: Settings → Accelerator → **GPU T4 x2**)
2. Aşağıda `CONFIG` sətrində modeli seç
3. **Runtime → Run all**. Google Drive icazəsi soruşanda icazə ver
4. Sessiya kəsilsə, yenidən **Run all**: təlim qaldığı yerdən davam edir

İstəyə bağlı: checkpoint-lərin Hugging Face-də də saxlanması üçün config-də `hub_repo` var.
Onda Colab-da solda açar işarəsinə (Secrets) bas, `HF_TOKEN` adı ilə öz write token-ini əlavə et.
Kaggle-da iki GPU olanda təlim avtomatik `torchrun` ilə hər ikisində gedir (DDP).
Odlar-2-dən əvvəl iki GPU-nun işlədiyini yoxlamaq üçün: `CONFIG = "configs/sinaq_t4x2.yaml"` (~35 dəqiqə).


In [ ]:
# 1) Ayarlar
CONFIG = "configs/sinaq_t4x2.yaml"   # odlar2.yaml, odlar15.yaml, odlar1.yaml, sinaq_t4x2.yaml (iki GPU sınağı), tiny2.yaml
EXTRA  = []                       # məs. ["train.max_steps=20000"]

In [ ]:
# 2) Hazırlıq: GPU, Drive, kod
import os, subprocess, yaml
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout or "GPU tapılmadı!")
name = os.path.splitext(os.path.basename(CONFIG))[0]
# Mühit: Kaggle-ın yeni şəklində google.colab paketi də var, ona görə paketə yox, qovluqlara baxırıq
KAGGLE = os.path.exists("/kaggle/working")
COLAB = not KAGGLE and os.path.exists("/var/colab/hostname")
if COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/Odlar/runs/{name}"
    REPO = "/content/odlar-repo"
elif KAGGLE:
    OUT = f"/kaggle/working/runs/{name}"
    REPO = "/kaggle/working/odlar-repo"
else:
    OUT, REPO = os.path.abspath(f"runs/{name}"), os.path.abspath("odlar-repo")
if not os.path.exists(REPO):
    !git clone -q https://github.com/krmhuseyn-sys/odlar.git {REPO}
else:
    !cd {REPO} && git pull -q
%cd {REPO}
!pip install -q -r requirements-train.txt
# HF_TOKEN (istəyə bağlı): öz mühitinin Secrets bölməsindən götürülür, təlim prosesinə ötürülür
from odlar.hubutil import hf_token
if hf_token():
    os.environ["HF_TOKEN"] = hf_token(); print("HF_TOKEN tapıldı")
print("Nəticələr:", OUT)

In [ ]:
# 3) Təlim (data yoxdursa, əvvəlcə hazırlanır)
import torch
from odlar.config import load_config
cfg = load_config(CONFIG)
sets = [f"out_dir={OUT}"] + EXTRA
if COLAB and cfg["train"].get("hub_repo") and os.environ.get("HF_TOKEN"):
    # Colab: checkpoint.pt lokal diskdə, Drive-da yalnız model.pt (K8: Drive zibil qutusu kvotanı doldurmasın).
    # Sessiya bitsə, checkpoint Hugging Face-dən bərpa olunur.
    sets.append("train.ckpt_dir=/content/ckpt")
SETS = " ".join(sets)
ngpu = torch.cuda.device_count()
print("GPU sayı:", ngpu)
if ngpu > 1:
    !torchrun --standalone --nproc_per_node={ngpu} -m odlar.train {CONFIG} --set {SETS}
else:
    !python -m odlar.train {CONFIG} --set {SETS}

In [ ]:
# 4) Sınaq: mətn yazdır və standart testlər
!python -m odlar.generate {OUT} "Qış gecəsi idi" -n 2 --tokens 80
!python -m odlar.evaluate {OUT} --out {OUT}/eval.json

In [ ]:
# 5) Brauzer üçün ONNX (sayta qoymaq üçün). Fayllar OUT/web qovluğuna yazılır
!python -m odlar.export {OUT} --name {name} --to {OUT}/web